# Mestres do XGH — Aula 15
## Machine learning de verdade

**Shift + Enter** roda a célula.

Na **Aula 3** você escreveu um classificador de spam com `if`. Depois viu um modelo aprender a
mesma coisa sozinho, com 16 exemplos, e todo mundo achou mágica.

Hoje a gente faz a mesma coisa **com rigor** — e descobre os três jeitos de se enganar sozinho.

> A frase da aula: **acurácia alta é a mentira mais confortável que existe.**

---
# 0. Gerando o dado da aula

Rode esta célula primeiro. Ela gera dois arquivos:

- `mensagens.csv` — 400 mensagens do grupo do curso, marcadas como spam ou não
- `mar_santos.csv` — 800 dias de mar na Ponta da Praia, marcados se teve ressaca

Nada de download: o `random.seed(15)` garante que o dado sai **igual** na máquina de todo mundo.

In [ ]:
import random
import pandas as pd

random.seed(15)

# ---- mensagens do grupo (spam ou nao) ----
mensagens = []
for _ in range(400):
    spam = random.random() < 0.45
    if spam:
        tem_link      = 1 if random.random() < 0.72 else 0
        tem_promocao  = 1 if random.random() < 0.62 else 0
        qtd_emoji     = random.randint(0, 9)
        tamanho       = random.randint(70, 320)
        hora          = random.choice([1, 2, 3, 7, 9, 12, 14, 19, 22, 23])
        desconhecido  = 1 if random.random() < 0.66 else 0
    else:
        tem_link      = 1 if random.random() < 0.33 else 0
        tem_promocao  = 1 if random.random() < 0.22 else 0
        qtd_emoji     = random.randint(0, 6)
        tamanho       = random.randint(15, 240)
        hora          = random.choice([8, 9, 10, 11, 12, 13, 15, 17, 18, 19, 20, 21])
        desconhecido  = 1 if random.random() < 0.30 else 0
    # o moderador apaga o spam depois de ver. guarde esse nome.
    apagou = 1 if (spam if random.random() < 0.97 else not spam) else 0
    mensagens.append({
        "tem_link": tem_link,
        "tem_promocao": tem_promocao,
        "qtd_emoji": qtd_emoji,
        "tamanho": tamanho,
        "hora": hora,
        "remetente_desconhecido": desconhecido,
        "moderador_apagou": apagou,
        "spam": int(spam),
    })

pd.DataFrame(mensagens).to_csv("mensagens.csv", index=False)

# ---- dias de mar na Ponta da Praia (teve ressaca ou nao) ----
dias = []
com_ressaca = set(random.sample(range(800), 40))   # exatamente 5%
for i in range(800):
    ressaca = i in com_ressaca
    if ressaca:
        vento   = round(random.uniform(14, 52), 1)
        pressao = round(random.uniform(1000, 1018), 1)
        temp    = round(random.uniform(18, 26), 1)
    else:
        vento   = round(random.uniform(4, 46), 1)
        pressao = round(random.uniform(1002, 1026), 1)
        temp    = round(random.uniform(17, 28), 1)
    dias.append({
        "vento_kmh": vento,
        "pressao_hpa": pressao,
        "temp_agua": temp,
        "mes": (i % 12) + 1,
        "ressaca": int(ressaca),
    })

pd.DataFrame(dias).to_csv("mar_santos.csv", index=False)

print("mensagens.csv  — 400 linhas")
print("mar_santos.csv — 800 linhas")

---
# 1. O vocabulário certo

Quatro palavras. Com elas você entende qualquer tutorial de machine learning.

| Palavra | O que é | No nosso caso |
|---|---|---|
| **feature** | uma informação de entrada, uma coluna | `tem_link`, `hora`, `tamanho` |
| **rótulo** | a resposta que você quer prever | `spam` (0 ou 1) |
| **treino** | os exemplos que o modelo pode ver | 300 mensagens |
| **teste** | os exemplos que o modelo **não** pode ver | 100 mensagens |

Em inglês você vai ver `X` para as features e `y` para o rótulo. É só isso.

Olha o dado:

In [ ]:
import pandas as pd

msg = pd.read_csv("mensagens.csv")
print(msg.head(5).to_string())
print()
print("linhas e colunas:", msg.shape)
print()
print(msg["spam"].value_counts())

Sete features, um rótulo. E repare: **tudo é número**.

O modelo não lê texto. Na Aula 3 você também não contava letras — contava `tem_link`,
`tem_promocao`. A diferença é que lá **você** escolheu o peso de cada uma.

---
# 2. Separar treino de teste

Aqui entra a biblioteca. Da **Aula 11** você já sabe a pegadinha:

```
pip install scikit-learn      <- o nome que instala
import sklearn                 <- o nome que importa
```

Agora a separação. Três linhas:

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["tem_link", "tem_promocao", "qtd_emoji",
            "tamanho", "hora", "remetente_desconhecido"]

X = msg[FEATURES]      # as features
y = msg["spam"]        # o rotulo

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=15, stratify=y)

print("treino:", X_treino.shape)
print("teste :", X_teste.shape)

- `test_size=0.25` — um quarto fica de fora
- `random_state=15` — o sorteio é o mesmo toda vez (sem isso, número diferente a cada execução)
- `stratify=y` — mantém a mesma proporção de spam nos dois lados

**Essas 100 mensagens do teste o modelo nunca vai ver no treino.** É o único jeito de saber se ele
aprendeu ou só decorou.

---
# 3. Antes de treinar: a regra que você escreveu na Aula 3

Na Aula 3 era um `if` com um limiar que você chutou. Vamos medir essa regra **nas mesmas 100
mensagens de teste** — assim tudo nesta aula é comparável.

In [ ]:
# a regra ingenua da Aula 3: se tem link E fala de promocao, e spam
regra_ingenua = ((X_teste["tem_link"] == 1) & (X_teste["tem_promocao"] == 1)).astype(int)
print("regra ingenua       :", (regra_ingenua == y_teste).mean())

# a mesma regra, ajustada na mao: pontos, e de 3 pra cima e spam
pontos = X_teste["tem_link"]*2 + X_teste["tem_promocao"]*2 + X_teste["remetente_desconhecido"]
regra_pontos = (pontos >= 3).astype(int)
print("regra de pontos     :", (regra_pontos == y_teste).mean())

Guarde esses dois números. São o **chão** da aula.

Machine learning não é mágica: é *não ter que inventar esse limiar na mão*. O computador procura
o corte sozinho, no dado. É um `if` **cujo limiar ninguém escreveu**.

---
# 4. 💥 Erro nº 1 — testar no dado do treino

Treine e meça **no próprio treino**. É o que todo mundo faz na primeira semana.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

modelo = DecisionTreeClassifier(random_state=15)
modelo.fit(X_treino, y_treino)

previsao_treino = modelo.predict(X_treino)
print("acuracia NO TREINO:", accuracy_score(y_treino, previsao_treino))

**100%.** Pode comemorar? Agora mede no teste — nas 100 mensagens que ele nunca viu:

In [ ]:
previsao_teste = modelo.predict(X_teste)
print("acuracia NO TESTE :", accuracy_score(y_teste, previsao_teste))

```
acuracia NO TREINO: 1.0
acuracia NO TESTE : 0.81
```

### Decorou o simulado

Imagine um aluno que recebe o simulado com gabarito, decora as 300 respostas e tira 10.
Na prova, com questões novas, tira 8,1.

Ele não aprendeu a matéria. Ele **decorou o simulado**. Esse é o nome do problema:
**overfitting**.

O número que importa é sempre o do **teste**. E o que denuncia o problema é a **distância entre os
dois**: 19 pontos de diferença é sinal de decoreba.

Veja a árvore que ele construiu:

In [ ]:
print("profundidade da arvore:", modelo.get_depth())
print("folhas (regras no fim):", modelo.get_n_leaves())

46 folhas para 300 exemplos. Ele praticamente fez uma gavetinha para cada mensagem.

### E se a gente deixar ele decorar menos?

`max_depth` limita o tamanho da árvore. Menos espaço para gavetinha:

In [ ]:
raso = DecisionTreeClassifier(max_depth=5, random_state=15)
raso.fit(X_treino, y_treino)

print("treino:", round(accuracy_score(y_treino, raso.predict(X_treino)), 4))
print("teste :", round(accuracy_score(y_teste,  raso.predict(X_teste)),  4))

```
treino: 0.8967
teste : 0.83
```

O modelo ficou **pior no treino** e **melhor no teste**. A distância caiu de 19 para 7 pontos.

> Modelo que acerta tudo no treino quase sempre está mentindo para você.

---
# 5. Baseline: com o que você está comparando?

83% parece bom. **Bom comparado a quê?**

O `DummyClassifier` é o modelo mais burro possível: ele olha o treino, vê qual classe aparece mais
e responde **sempre aquela**. Nem olha as features.

In [ ]:
from sklearn.dummy import DummyClassifier

burro = DummyClassifier(strategy="most_frequent")
burro.fit(X_treino, y_treino)
print("chutar sempre a classe maior:", accuracy_score(y_teste, burro.predict(X_teste)))

Agora a escada completa deste problema, no **mesmo** conjunto de teste:

| Quem decide | Acurácia no teste |
|---|---|
| chutar sempre a classe maior (`DummyClassifier`) | **0.51** |
| a regra ingênua da Aula 3 (`link` **e** `promoção`) | **0.72** |
| a regra de pontos, ajustada na mão | **0.80** |
| a árvore de decisão (`max_depth=5`) | **0.83** |

Leia isso com calma, porque é desconfortável: **o modelo ganhou da sua regra por 3 pontos.**

Então para que serve? Para três coisas:

1. você **não precisou inventar** o limiar
2. quando o spam mudar, você **retreina** — não reescreve o `if`
3. e, principalmente, agora você tem um **número honesto** para dizer se melhorou

> Baseline não é formalidade. É a diferença entre *"meu modelo acerta 83%"* e
> *"meu modelo acerta 83% onde chutar acerta 51%"*. A segunda frase é informação.

---
# 6. 💥 Erro nº 2 — a acurácia de 95% que não vale nada

Troca de problema. Agora é o mar da Ponta da Praia: **vai ter ressaca amanhã?**

Ressaca é raro. Olhe o dado antes de qualquer coisa — é o hábito da **Aula 10**:

In [ ]:
mar = pd.read_csv("mar_santos.csv")
print(mar.head(5).to_string())
print()
print(mar["ressaca"].value_counts())
print()
print(mar["ressaca"].value_counts(normalize=True))

**95% dos dias não tem ressaca.** Agora o modelo mais burro do mundo, sem olhar feature nenhuma:

In [ ]:
X_mar = mar[["vento_kmh", "pressao_hpa", "temp_agua", "mes"]]
y_mar = mar["ressaca"]

Xm_treino, Xm_teste, ym_treino, ym_teste = train_test_split(
    X_mar, y_mar, test_size=0.25, random_state=15, stratify=y_mar)

burro_mar = DummyClassifier(strategy="most_frequent")
burro_mar.fit(Xm_treino, ym_treino)
print("acuracia do modelo BURRO:", accuracy_score(ym_teste, burro_mar.predict(Xm_teste)))

```
acuracia do modelo BURRO: 0.95
```

**95%.** Esse modelo responde *"não vai ter ressaca"* para absolutamente qualquer dia. Para sempre.
Não olhou vento, não olhou pressão, não olhou nada.

Se alguém te mandar isso num e-mail — *"treinei um modelo e deu 95% de acurácia"* — a primeira
pergunta não é *"que modelo?"*. É: **quanto dá chutando?**

Agora o modelo de verdade, no mesmo dado:

In [ ]:
modelo_mar = DecisionTreeClassifier(max_depth=4, random_state=15)
modelo_mar.fit(Xm_treino, ym_treino)

print("acuracia do modelo de verdade:", accuracy_score(ym_teste, modelo_mar.predict(Xm_teste)))

96% contra 95%. **Um ponto.** Todo o trabalho para um ponto acima de chutar.

### A matriz de confusão denuncia o que a acurácia esconde

A acurácia é **um** número e ele mistura tudo. A matriz de confusão mostra os quatro casos
separados: acertou o não, errou o não, acertou o sim, errou o sim.

In [ ]:
from sklearn.metrics import confusion_matrix

def mostrar(titulo, modelo_qualquer):
    mat = confusion_matrix(ym_teste, modelo_qualquer.predict(Xm_teste))
    print(titulo)
    print(pd.DataFrame(mat,
                       index=["real: SEM ressaca", "real: COM ressaca"],
                       columns=["previu SEM", "previu COM"]))
    print()

mostrar("--- modelo BURRO ---", burro_mar)
mostrar("--- modelo de verdade ---", modelo_mar)

```
--- modelo BURRO ---
                   previu SEM  previu COM
real: SEM ressaca         190           0
real: COM ressaca          10           0

--- modelo de verdade ---
                   previu SEM  previu COM
real: SEM ressaca         189           1
real: COM ressaca           7           3
```

**Olha a coluna da direita do modelo burro: zero.** Ele nunca, nem uma vez, avisou que ia ter
ressaca. Teve 10 ressacas no teste e ele perdeu todas as 10 — com 95% de acurácia.

O modelo de verdade pegou 3 das 10. Pouco, mas **existe**. E a acurácia dos dois é quase igual.

Se esse modelo fosse o alerta da Defesa Civil, o de 95% deixaria as dez ressacas passarem sem
avisar ninguém.

### O relatório que o sklearn já te dá de graça

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(ym_teste, burro_mar.predict(Xm_teste), zero_division=0))

Não precisa entender `precision` e `f1-score` hoje. Olhe **uma** coluna: `recall` da linha `1`.

`recall` é *"dos dias que tiveram ressaca de verdade, quantos eu peguei?"*

No modelo burro: **0.00**. Zero por cento. Com 95% de acurácia na mesma tabela.

> Em dado desbalanceado, acurácia é quase inútil. Quando o que você procura é raro —
> fraude, doença, defeito, ressaca — o número que importa é quantos dos raros você pegou.

---
# 7. 💥 Erro nº 3 — vazamento de dado

Esse é o mais caro e o menos comentado. E é fácil de cair sem perceber.

Volte às mensagens. Tem uma coluna que a gente deixou de fora de propósito:
**`moderador_apagou`**.

Parece uma feature legítima, não parece? É informação sobre a mensagem, está na tabela,
é número. Vamos colocar.

In [ ]:
FEATURES_VAZADAS = FEATURES + ["moderador_apagou"]

Xv = msg[FEATURES_VAZADAS]
Xv_treino, Xv_teste, yv_treino, yv_teste = train_test_split(
    Xv, y, test_size=0.25, random_state=15, stratify=y)

vazado = DecisionTreeClassifier(random_state=15)
vazado.fit(Xv_treino, yv_treino)

print("acuracia no treino:", accuracy_score(yv_treino, vazado.predict(Xv_treino)))
print("acuracia no TESTE :", accuracy_score(yv_teste,  vazado.predict(Xv_teste)))

```
acuracia no treino: 1.0
acuracia no TESTE : 1.0
```

**100% no teste.** Não é overfitting — o teste é dado que ele nunca viu. Fizemos tudo certo:
separamos treino e teste, usamos `stratify`, medimos no lado de fora.

E o número é perfeito. Pergunte quem fica feliz com isso.

Veja de onde ele tirou a resposta:

In [ ]:
importancia = pd.Series(vazado.feature_importances_, index=FEATURES_VAZADAS)
print(importancia.sort_values(ascending=False).round(3))

```
moderador_apagou          0.909
hora                      0.036
tamanho                   0.032
tem_link                  0.014
remetente_desconhecido    0.007
tem_promocao              0.002
qtd_emoji                 0.000
dtype: float64
```

91% da decisão vem de **uma** coluna.

### Por que isso é inútil na vida real

Pense em **quando** cada informação existe.

O moderador só apaga a mensagem **depois** de ler e concluir que é spam. Ou seja:
`moderador_apagou` **é a resposta**, escrita com outro nome.

O modelo não aprendeu a reconhecer spam. Ele aprendeu a copiar o moderador.

E na hora de usar de verdade — mensagem nova, acabou de chegar no grupo — essa coluna **não
existe ainda**. É justamente a decisão que você queria que o modelo tomasse.

Olha o que acontece quando você tenta usar:

In [ ]:
# mensagem nova: o moderador ainda nao viu, entao essa coluna nao existe
mensagem_nova = pd.DataFrame([{
    "tem_link": 1, "tem_promocao": 1, "qtd_emoji": 4,
    "tamanho": 180, "hora": 2, "remetente_desconhecido": 1,
}])

vazado.predict(mensagem_nova)

```
ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- moderador_apagou
```

**Leia a mensagem.** O sklearn está dizendo exatamente o problema: o modelo quer uma coluna que
você não tem. E você nunca vai ter, no momento em que precisa prever.

Um modelo de 100% de acurácia que **não roda** em produção. É esse o preço do vazamento.

A versão honesta, sem a coluna vazada, dá **0.81** — e essa você pode usar.

### Como se proteger

Uma pergunta por feature, e só essa:

> **No momento em que eu preciso prever, essa informação já existe?**

Exemplos de vazamento que acontecem de verdade:

| Você quer prever | A feature traiçoeira | Por quê |
|---|---|---|
| se o cliente vai cancelar | `data_do_cancelamento` | só existe depois de cancelar |
| se o paciente tem a doença | `remedio_prescrito` | o médico já diagnosticou |
| se o aluno vai desistir | `nota_final` | vem no fim do semestre |
| se a peça vai quebrar | `custo_do_reparo` | o reparo já aconteceu |

E a regra de bolso que salva:

> **Acurácia suspeita de boa é bug, não é talento.** Quando der perto de 100%, procure o vazamento
> antes de comemorar. Em dado real, 100% não existe.

---
# 8. 💥 Bônus: dois erros que você vai ver na terça

Esses dois não são erros de raciocínio, são erros de teclado. Mas tomam uma hora se você não
reconhecer a mensagem.

### O primeiro é a pegadinha da Aula 11

In [ ]:
import scikit_learn

```
ModuleNotFoundError: No module named 'scikit_learn'
```

Instala com `scikit-learn`, importa como `sklearn`. Era a primeira linha da tabela da Aula 11.

E se você tentar com hífen fica pior, porque nem chega a rodar:

```python
import scikit-learn
```

```
SyntaxError: invalid syntax
```

Hífen não pode em nome de variável — e `import` espera um nome de variável.

### O segundo: o modelo não lê texto

In [ ]:
textos = pd.DataFrame({
    "texto": ["ganhe agora", "bom dia", "clique aqui", "cheguei"],
    "tem_link": [1, 0, 1, 0],
})
rotulos = [1, 0, 1, 0]

DecisionTreeClassifier().fit(textos, rotulos)

```
ValueError: could not convert string to float: 'ganhe agora'
```

Toda feature precisa ser número. Transformar texto em número tem nome, tem técnica, e é a
**Aula 19** — por enquanto, se a coluna é texto, ela não entra direto.

---
# 9. 🛠️ Mão na massa — classificador de ponta a ponta

Sozinho, do zero, no dado das mensagens. Seis passos, e o relatório no fim.

1. ler `mensagens.csv`
2. separar `X` (features **honestas**, sem `moderador_apagou`) e `y`
3. `train_test_split` com `test_size=0.3` e `random_state=42`
4. treinar um `DecisionTreeClassifier(max_depth=4, random_state=42)`
5. medir acurácia **no treino** e **no teste**
6. medir o `DummyClassifier` no mesmo teste

E no fim imprimir três frases:

```
meu modelo no teste:  0.??
chutar a classe maior: 0.??
distancia treino-teste: ?? pontos
```

**Perguntas para responder em voz alta quando terminar:**

- ganhou do baseline? por quanto?
- a distância treino-teste é grande? ele decorou?
- mudar o `random_state` muda a acurácia? **mude e veja** — isso ensina mais que a aula inteira

In [ ]:
# Seu espaco — passos 1 a 6


<details>
<summary>Clique só depois de tentar</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

msg = pd.read_csv("mensagens.csv")

FEATURES = ["tem_link", "tem_promocao", "qtd_emoji",
            "tamanho", "hora", "remetente_desconhecido"]
X, y = msg[FEATURES], msg["spam"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
d = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

a_tr = accuracy_score(y_tr, m.predict(X_tr))
a_te = accuracy_score(y_te, m.predict(X_te))

print("meu modelo no teste:  ", round(a_te, 4))
print("chutar a classe maior:", round(accuracy_score(y_te, d.predict(X_te)), 4))
print("distancia treino-teste:", round((a_tr - a_te) * 100, 1), "pontos")
```
</details>

---
# 10. O que isso destrava

Você não treinou um modelo melhor hoje. Você ganhou o direito de **não acreditar** num número.

- consegue **ler** uma métrica que alguém te mostra e saber qual pergunta fazer
- consegue dizer *"isso é mais ou menos que chutar?"* — e é a pergunta que ninguém faz
- reconhece vazamento de dado, que é o erro que mais se vê em projeto de empresa
- e tem o vocabulário para ler qualquer tutorial: feature, rótulo, treino, teste, baseline

> Na Aula 10 o dado sujo derrubou um modelo de 100% para 25% e **não deu erro nenhum**.
> Hoje o dado estava limpo e o modelo mentiu de três jeitos diferentes — também sem dar erro.
> Machine learning não quebra: ele te dá um número errado com cara de certo.

---
# 11. 📝 Tarefa da semana

**1.** Rode a mão na massa e me mande no grupo **as três frases** do relatório final.

**2.** Mexa no `max_depth` do modelo das mensagens: teste `1`, `3`, `5`, `10` e `None`.
Monte uma tabelinha com treino e teste de cada um e me diga **onde começa a decoreba**.

**3.** Volte ao `mar_santos.csv` e tente fazer o modelo pegar **mais de 3** das 10 ressacas.
Dica: `DecisionTreeClassifier(class_weight="balanced")`. Olhe a matriz de confusão, não a
acurácia — e repare no preço que você paga nos falsos alarmes.

**4.** Caça ao vazamento: pense num problema **do seu trabalho** que você gostaria de prever.
Liste 5 features. Agora passe a pergunta em cada uma: *no momento de prever, essa informação já
existe?* Mande no grupo a que não passou — todo mundo tem uma.

---

## A pergunta que fica

A gente usou `DecisionTreeClassifier` a aula inteira e tratou como caixa preta: entra feature,
sai resposta.

**Mas como ele decide por dentro?** Por que "árvore"? Onde ele achou o limiar que você não escreveu?

E existem outros modelos — por que eu escolheria um e não o outro?

*(Aula 16: primeiro modelo com `sklearn`, por dentro.)*

---

*Mestres do XGH · Aula 15 · 21/11/2026*

In [ ]:
# Tarefa 2 — a tabelinha de max_depth
